# Лабораторная работа №3
## Многоагентные системы (Multi-Agent Systems)

**Студент:** Мыльников Александр Русланович

**Группа:** ФИТ-221

**Тема диплома:** Разработка системы неразрушающего контроля для выявления дефектов металлических бутылок на конвейерной линии

## 1. Импорт библиотек и настройка окружения

In [ ]:
import sys
import os
import logging
from datetime import datetime

# Добавляем путь к src
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# Настройка логирования
logging.basicConfig(level=logging.ERROR)

print("Библиотеки загружены")
print(f"Текущая директория: {os.getcwd()}")

## 2. Проверка Yandex GPT (интеграция с LLM)

In [ ]:
from src.llm.yandex_gpt_client import yandex_gpt

print("=" * 60)
print("ПРОВЕРКА YANDEX GPT")
print("=" * 60)

if yandex_gpt.is_available():
    print("✅ Yandex GPT доступен")
    test_response = yandex_gpt.generate("Что такое неразрушающий контроль? Напиши одно предложение.")
    print(f"📝 Тестовый ответ: {test_response}")
else:
    print("⚠️ Yandex GPT не настроен, будут использованы моки")

print("=" * 60)

## 3. Создание базовых агентов

In [ ]:
from src.agents.researcher_agent import ResearcherAgent
from src.agents.analyst_agent import AnalystAgent
from src.agents.writer_agent import WriterAgent

# Создание агентов
researcher = ResearcherAgent()
analyst = AnalystAgent()
writer = WriterAgent()

print("=" * 60)
print("СОЗДАННЫЕ АГЕНТЫ")
print("=" * 60)
print(f"🔍 {researcher.config.role}")
print(f"📊 {analyst.config.role}")
print(f"✍️ {writer.config.role}")
print("=" * 60)

## 4. Проверка возможностей агентов

In [ ]:
print("ВОЗМОЖНОСТИ АГЕНТОВ:")
print("-" * 40)

print("\n🔍 ResearcherAgent:")
for cap in researcher.get_capabilities():
    print(f"   • {cap}")

print("\n📊 AnalystAgent:")
for cap in analyst.get_capabilities():
    print(f"   • {cap}")

print("\n✍️ WriterAgent:")
for cap in writer.get_capabilities():
    print(f"   • {cap}")

## 5. Одиночные задачи агентов

In [ ]:
import time

print("=" * 60)
print("ВЫПОЛНЕНИЕ ОДИНОЧНЫХ ЗАДАЧ")
print("=" * 60)

# Задача для исследования
topic = "Методы неразрушающего контроля металлических бутылок"
print(f"\n📌 ТЕМА: {topic}\n")

# 1. ResearcherAgent
print("🔍 1. ИССЛЕДОВАТЕЛЬ:")
print("-" * 40)
start = time.time()
research_result = researcher.execute_task(topic)
print(f"   Статус: {research_result['status']}")
print(f"   Время: {research_result['execution_time']:.2f}с")
print(f"   Источников: {len(research_result.get('sources_found', []))}")
print(f"\n   🔑 Ключевые факты:")
for i, fact in enumerate(research_result.get('key_facts', [])[:3], 1):
    print(f"      {i}. {fact[:100]}...")
print(f"\n   📝 Рекомендации:")
for rec in research_result.get('recommendations', [])[:2]:
    print(f"      • {rec}")

In [ ]:
# 2. AnalystAgent
print("\n📊 2. АНАЛИТИК:")
print("-" * 40)
start = time.time()
analysis_result = analyst.execute_task(topic, {"research_data": research_result})
print(f"   Статус: {analysis_result['status']}")
print(f"   Время: {analysis_result['execution_time']:.2f}с")
print(f"\n   💡 Инсайты:")
for i, insight in enumerate(analysis_result.get('insights', [])[:3], 1):
    print(f"      {i}. {insight}")
print(f"\n   📈 Метрики:")
metrics = analysis_result.get('metrics', {})
for key, value in metrics.items():
    print(f"      • {key}: {value}")

In [ ]:
# 3. WriterAgent
print("\n✍️ 3. ПИСАТЕЛЬ:")
print("-" * 40)
start = time.time()
writer_result = writer.execute_task(topic, {
    "research_data": research_result,
    "analysis_data": analysis_result
})
print(f"   Статус: {writer_result['status']}")
print(f"   Время: {writer_result['execution_time']:.2f}с")
print(f"   Длина документа: {len(writer_result.get('document', ''))} символов")
print(f"\n   📄 Документ (первые 500 символов):")
print("   " + "-" * 50)
print(writer_result.get('document', '')[:500])
print("   " + "-" * 50)

## 6. Координация агентов через Crew

In [ ]:
from src.crew.research_crew import ResearchCrew

print("=" * 60)
print("КООРДИНАЦИЯ АГЕНТОВ ЧЕРЕЗ CREW")
print("=" * 60)

# Создание команды
crew = ResearchCrew()
print("\n✅ Команда агентов создана")

# Выполнение комплексной задачи
task = "Современные методы неразрушающего контроля металлических бутылок"
print(f"\n📌 ЗАДАЧА: {task}\n")

print("🔄 Процесс выполнения:")
print("   ResearcherAgent (исследование) → AnalystAgent (анализ) → WriterAgent (отчёт)\n")

start = time.time()
crew_result = crew.execute(task)
total_time = time.time() - start

print(f"✅ Статус: {'УСПЕШНО' if crew_result.success else 'ОШИБКА'}")
print(f"⏱️ Общее время: {total_time:.2f}с")
print(f"🤖 Участвовало агентов: {len(crew_result.agent_results)}")
print(f"📄 Длина отчёта: {len(crew_result.final_output)} символов")

In [ ]:
print("\n" + "=" * 60)
print("ФИНАЛЬНЫЙ ОТЧЁТ")
print("=" * 60)
print(crew_result.final_output)
print("=" * 60)

## 7. Специализированный агент для диплома (NDTInspectorAgent)

In [ ]:
from src.ndt_inspector_agent import NDTInspectorAgent

print("=" * 60)
print("СПЕЦИАЛИЗИРОВАННЫЙ АГЕНТ (НЕРАЗРУШАЮЩИЙ КОНТРОЛЬ)")
print("=" * 60)

# Создание агента
ndt_inspector = NDTInspectorAgent()
print(f"\n✅ Агент создан")
print(f"   Роль: {ndt_inspector.config.role}")
print(f"   Цель: {ndt_inspector.config.goal}")

print("\n🔧 ВОЗМОЖНОСТИ:")
for cap in ndt_inspector.get_capabilities():
    print(f"   • {cap}")

In [ ]:
print("\n" + "=" * 60)
print("ИНСПЕКЦИЯ МЕТАЛЛИЧЕСКИХ БУТЫЛОК")
print("=" * 60)

bot  tle_results = []
print("\n📊 РЕЗУЛЬТАТЫ ИНСПЕКЦИЙ:")
print("-" * 55)

for i in range(1, 11):
    result = ndt_inspector.execute_task(
        "Контроль качества бутылки",
        context={"bottle_id": f"BOTTLE_{i:03d}"}
    )
    bottle_results.append(result)
    
    inspection = result.get('inspection_result', {})
    decision = result.get('quality_decision', {})
    
    quality = inspection.get('quality_score', 0)
    status_icon = "❌" if decision.get('reject') else "✅"
    status_text = "БРАК" if decision.get('reject') else "ГОДЕН"
    
    print(f"   {status_icon} Бутылка {i:3d}: {status_text:6s} | Качество: {quality:5.1f}% | {decision.get('action', 'N/A')}")

print("-" * 55)

In [ ]:
# Статистика инспекций
stats = ndt_inspector.get_statistics_full()

print("\n📈 СТАТИСТИКА ИНСПЕКЦИЙ:")
print(f"   Всего проверено: {stats['total']}")
print(f"   Годных: {stats['accept']}")
print(f"   Бракованных: {stats['defective']}")
print(f"   Процент годных: {stats['accept_rate']:.1f}%")
print(f"   Среднее качество: {stats['avg_quality_score']:.1f}%")

# Детализация по типам дефектов
if stats.get('defects_distribution'):
    print("\n📊 РАСПРЕДЕЛЕНИЕ ДЕФЕКТОВ:")
    for defect_type, count in stats['defects_distribution'].items():
        print(f"   • {defect_type}: {count}")

print("\n" + "=" * 60)

## 8. Шина сообщений (Message Bus)

In [ ]:
from src.communication.message_bus import MessageBus

print("=" * 60)
print("ТЕСТИРОВАНИЕ ШИНЫ СООБЩЕНИЙ")
print("=" * 60)

# Создание шины
bus = MessageBus()
received_messages = []

def callback(message):
    received_messages.append(message)
    print(f"   📬 Получено сообщение: {message}")

# Подписка
sub_id = bus.subscribe("test_event", callback)
print(f"\n✅ Подписка создана: {sub_id[:8]}...")

# Публикация сообщений
print("\n📤 Публикация сообщений:")
bus.publish({"type": "test_event", "data": "Сообщение 1"})
bus.publish({"type": "test_event", "data": "Сообщение 2"})

print(f"\n📊 Получено сообщений: {len(received_messages)}")

# Отписка
bus.unsubscribe(sub_id)
print("\n✅ Отписка выполнена")

# Проверка отписки
bus.publish({"type": "test_event", "data": "Сообщение после отписки"})
print(f"📊 Получено после отписки: {len(received_messages)} (новых нет)")

print("\n" + "=" * 60)

## 9. Интеграция с дипломом

### Демонстрация работы системы на реальных данных

In [ ]:
print("=" * 60)
print("ИНТЕГРАЦИЯ С ДИПЛОМОМ")
print("=" * 60)

# Сценарий: инспекция партии бутылок с автоматическим отчётом
print("\n🏭 СЦЕНАРИЙ: Автоматическая инспекция партии бутылок\n")

# 1. Инспекция партии
batch_results = []
batch_size = 20

print("🔍 Проводится инспекция партии из 20 бутылок...\n")

for i in range(1, batch_size + 1):
    result = ndt_inspector.execute_task(
        "Контроль качества",
        context={"bottle_id": f"BATCH_{i:03d}"}
    )
    batch_results.append(result)

# 2. Анализ партии
good_count = sum(1 for r in batch_results if not r['quality_decision']['reject'])
defect_count = batch_size - good_count
avg_quality = sum(r['inspection_result']['quality_score'] for r in batch_results) / batch_size

print("📊 РЕЗУЛЬТАТЫ ПАРТИИ:")
print(f"   Всего бутылок: {batch_size}")
print(f"   Годных: {good_count}")
print(f"   Бракованных: {defect_count}")
print(f"   Процент годных: {good_count/batch_size*100:.1f}%")
print(f"   Среднее качество: {avg_quality:.1f}%")

# 3. Генерация производственного отчёта через Crew
print("\n📄 ГЕНЕРАЦИЯ ПРОИЗВОДСТВЕННОГО ОТЧЁТА...\n")

production_task = f"""
Производственный отчёт по результатам неразрушающего контроля партии металлических бутылок.

Статистика:
- Проверено бутылок: {batch_size}
- Годных: {good_count}
- Бракованных: {defect_count}
- Процент годных: {good_count/batch_size*100:.1f}%
- Среднее качество: {avg_quality:.1f}%

Подготовь отчёт для руководства.
"""

production_report = crew.execute(production_task)

if production_report.success:
    print("✅ Производственный отчёт сгенерирован")
    print(f"   Длина отчёта: {len(production_report.final_output)} символов")

print("\n" + "=" * 60)

In [ ]:
# Показать часть производственного отчёта
print("\n📄 ФРАГМЕНТ ПРОИЗВОДСТВЕННОГО ОТЧЁТА:")
print("-" * 60)
print(production_report.final_output[:1000])
if len(production_report.final_output) > 1000:
    print("\n... (отчёт обрезан для отображения)")
print("-" * 60)

## 10. Статистика и мониторинг

In [ ]:
print("=" * 60)
print("СТАТИСТИКА РАБОТЫ СИСТЕМЫ")
print("=" * 60)

# Статистика Crew
crew_stats = crew.get_statistics()
print("\n📊 CREW СТАТИСТИКА:")
print(f"   Выполнено задач: {crew_stats['statistics']['crews_executed']}")
print(f"   Успешно: {crew_stats['statistics']['successful_executions']}")
print(f"   Процент успеха: {crew_stats['success_rate']:.1f}%")

# Статистика NDT инспектора
ndt_stats = ndt_inspector.get_statistics_full()
print("\n🔍 NDT ИНСПЕКТОР:")
print(f"   Всего инспекций: {ndt_stats['total']}")
print(f"   Годных: {ndt_stats['accept']}")
print(f"   Бракованных: {ndt_stats['defective']}")
print(f"   Процент годных: {ndt_stats['accept_rate']:.1f}%")
print(f"   Среднее качество: {ndt_stats['avg_quality_score']:.1f}%")

# Статистика Yandex GPT
if yandex_gpt.is_available():
    print("\n🤖 YANDEX GPT:")
    print("   Статус: Доступен")
    print("   Режим: Реальные запросы к API")
else:
    print("\n🤖 YANDEX GPT:")
    print("   Статус: Не настроен")
    print("   Режим: Мок-данные")

print("\n" + "=" * 60)

## 11. Итоговый вывод

In [ ]:
print("=" * 60)
print("ИТОГИ ЛАБОРАТОРНОЙ РАБОТЫ №3")
print("=" * 60)

summary = {
    "✅ ResearcherAgent": "Исследователь - сбор информации через Yandex GPT",
    "✅ AnalystAgent": "Аналитик - анализ данных и выявление инсайтов",
    "✅ WriterAgent": "Писатель - генерация отчётов",
    "✅ NDTInspectorAgent": "Специализированный агент для НК металлических бутылок",
    "✅ ResearchCrew": "Координация последовательного выполнения задач",
    "✅ MessageBus": "Асинхронная коммуникация между агентами",
    "✅ Yandex GPT": "Интеграция с LLM для генерации ответов",
    "✅ Тесты": "5/5 тестов пройдены успешно"
}

print("\n📋 ВЫПОЛНЕННЫЕ ЗАДАЧИ:")
for task, description in summary.items():
    print(f"   {task}: {description}")

print("\n🎓 РАБОТА ГОТОВА К СДАЧЕ")
print("=" * 60)